In [5]:
"""
GPU Post-Training Quantization (PTQ) — ResNet-50 / CIFAR-10
=============================================================
GPU-native PTQ using multiple frameworks, since PyTorch's static PTQ
is CPU-only. This script benchmarks three GPU-capable approaches:

  1. ONNX Runtime  — export FP32 → ONNX → INT8 via ORT quantization API
                     (CUDAExecutionProvider for inference)
  2. TensorRT      — export FP32 → ONNX → TRT INT8 engine with calibration
                     (requires tensorrt + pycuda; best GPU throughput)
  3. Torch CUDA    — dynamic PTQ (Linear → qint8) run on CUDA tensors;
                     PT2E (PyTorch 2 Export) + X86InductorQuantizer → torch.compile

FLOPs measurement:
  - fvcore.nn.FlopCountAnalysis for FP32/quantized PyTorch models
  - Manual formula for conv/linear: FLOPs = 2 × Cin × Cout × Kh × Kw × H × W
  - IMPORTANT: FLOPs (operation count) are IDENTICAL for FP32 and INT8 models.
    Quantization does not reduce the number of operations — it changes the
    datatype. The hardware throughput advantage of INT8 is ~4× on Ampere Tensor
    Cores because each SIMD lane processes 4× more INT8 elements per cycle.
    FLOPs and throughput are different quantities and must not be conflated.

Mathematical foundation (same as CPU PTQ, different execution backend):
  Q(x)  = round(x / S) + Z              [quantization]
  x̂     = S · (Q(x) − Z)               [dequantization]
  ε     = x − x̂,  |ε| ≤ S/2           [bounded error]
  S     = (x_max − x_min) / (2^b − 1)  [scale — from calibration data]
  Z     = round(−x_min / S)             [zero-point]

GPU throughput advantage (hardware parallelism, not fewer operations):
  - INT8 Tensor Core GEMM: ~4× throughput vs FP32 on Ampere (A100, RTX 30xx)
  - INT8 Tensor Core GEMM: ~8× throughput vs FP32 on Hopper (H100)
  - Memory bandwidth: ~4× reduction (1 byte INT8 vs 4 bytes FP32 per weight)
  - TensorRT layer fusion: Conv+BN+ReLU → single kernel, fewer memory round-trips
  - ORT CUDAExecutionProvider: uses cuDNN (Conv), cuBLASLt (MatMul), and custom
    kernels depending on op type and graph structure — not all ops use cuDNN INT8.

Output: gpu_ptq_results.json
"""

# ── Imports ────────────────────────────────────────────────────────────────────
import os
import sys
import json
import time
import copy
import math
import tempfile
import warnings
import argparse
import subprocess
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np

warnings.filterwarnings("ignore")

# ── Config ─────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "gpu_ptq_results_v3.json"

BATCH_SIZE     = 128
NUM_WORKERS    = 4
NUM_CLASSES    = 10
CALIB_SIZE     = 1024
CALIB_BATCHES  = 8
INFERENCE_RUNS = 50
INPUT_SHAPE    = (1, 3, 32, 32)   # single-image shape for export/FLOPs

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

# ── Dependency detection ───────────────────────────────────────────────────────
def _try_import(name: str):
    try:
        import importlib
        return importlib.import_module(name)
    except ImportError:
        return None

torch         = _try_import("torch")
torchvision   = _try_import("torchvision")
ort           = _try_import("onnxruntime")
onnx          = _try_import("onnx")
onnxquant     = _try_import("onnxruntime.quantization")
fvcore        = _try_import("fvcore")
tensorrt      = _try_import("tensorrt")
pycuda        = _try_import("pycuda")
bitsandbytes  = _try_import("bitsandbytes")

# ── Install helpers (run once if needed) ───────────────────────────────────────
INSTALL_COMMANDS = {
    "torch":          "pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121",
    "onnx":           "pip install onnx onnxruntime-gpu",
    "onnxruntime":    "pip install onnxruntime-gpu",  # GPU build
    "fvcore":         "pip install fvcore",
    "tensorrt":       "pip install tensorrt",         # NVIDIA wheel
    "bitsandbytes":   "pip install bitsandbytes",
    "optimum":        "pip install optimum[onnxruntime-gpu]",
}

def install_deps():
    """Install all GPU PTQ dependencies. Run once before the experiment."""
    cmds = [
        # PyTorch with CUDA 12.1
        "pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121",
        # ONNX ecosystem (GPU runtime)
        "pip install onnx onnxruntime-gpu",
        # FLOPs counter
        "pip install fvcore",
        # TensorRT Python bindings (NVIDIA GPU required)
        "pip install tensorrt pycuda",
        # BitsAndBytes INT8/NF4 GPU quantization
        "pip install bitsandbytes",
        # Optimum for HuggingFace-style ONNX export + quantization
        "pip install optimum[onnxruntime-gpu]",
    ]
    for cmd in cmds:
        print(f"  $ {cmd}")
        subprocess.run(cmd.split(), check=False)


# ══════════════════════════════════════════════════════════════════════════════
# Model & Data
# ══════════════════════════════════════════════════════════════════════════════
def build_model(num_classes: int = 10):
    """ResNet-50 adapted for CIFAR-10 (32×32 input)."""
    import torch.nn as nn
    from torchvision import models
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path: str):
    model = build_model(NUM_CLASSES)
    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
    model.eval()
    return model

def get_test_loader():
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = datasets.CIFAR10(root="../data", train=False, download=True, transform=tf)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

def get_calib_loader():
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader, Subset
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds  = datasets.CIFAR10(root="../data", train=True, download=True, transform=tf)
    sub = Subset(ds, list(range(CALIB_SIZE)))
    return DataLoader(sub, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)


# ══════════════════════════════════════════════════════════════════════════════
# FLOPs Measurement
# ══════════════════════════════════════════════════════════════════════════════
def count_flops_fvcore(model, input_shape=INPUT_SHAPE) -> Dict:
    """
    Use fvcore for accurate FLOPs counting.
    fvcore counts multiply-accumulate ops (MACs); we report GFLOPs = 2×MACs / 1e9.

    For a Conv2d layer:
      MACs = Cout × Cin × Kh × Kw × Hout × Wout
      FLOPs = 2 × MACs  (one multiply + one accumulate per MAC)

    For a Linear layer:
      MACs = Cout × Cin
      FLOPs = 2 × MACs
    """
    from fvcore.nn import FlopCountAnalysis, parameter_count
    x = torch.randn(*input_shape)
    flops = FlopCountAnalysis(model, x)
    flops.unsupported_ops_warnings(False)
    total_macs = flops.total()
    params     = parameter_count(model)[""]
    return {
        "total_macs"   : int(total_macs),
        "total_flops"  : int(total_macs * 2),   # FLOPs = 2 × MACs
        "gflops_fp32"  : round(total_macs * 2 / 1e9, 4),
        "params_total" : int(params),
        "params_M"     : round(params / 1e6, 3),
        # ── Throughput context (NOT fewer FLOPs) ────────────────────────────
        # FLOPs are a property of the graph (op count), not the datatype.
        # INT8 and FP32 models perform the SAME number of multiply-accumulates.
        # The ~4× speedup on Ampere Tensor Cores comes from hardware-level
        # SIMD width: one INT8 GEMM instruction handles 4× more elements than
        # one FP32 instruction in the same cycle budget.
        "hardware_throughput_context": {
            "flops_unchanged_by_quantization": True,
            "int8_tensor_core_speedup_ampere": "~4× throughput vs FP32 (hardware parallelism)",
            "int8_tensor_core_speedup_hopper": "~8× throughput vs FP32 (hardware parallelism)",
            "memory_bandwidth_reduction"     : "~4× (1 byte INT8 vs 4 bytes FP32 per weight)",
            "correct_framing": (
                "Quantization reduces memory bandwidth and enables wider SIMD. "
                "It does NOT reduce FLOPs. Report wall-clock latency for real speedup."
            ),
        },
        "note": (
            "FLOPs = 2 × MACs (one multiply + one accumulate per MAC). "
            "FLOPs are IDENTICAL for INT8 and FP32 models — quantization does not "
            "change the op count. Use measured latency to assess actual GPU speedup."
        ),
    }

def count_flops_manual(model) -> Dict:
    """
    Manual FLOPs calculation by walking named modules.
    Formula per layer:
      Conv2d  : FLOPs = 2 × Cin × Cout × Kh × Kw × Hout × Wout × N
      Linear  : FLOPs = 2 × in_features × out_features × N
      BN      : FLOPs = 4 × C × H × W × N  (normalize, scale, shift, var)

    H_out = floor((H_in + 2p - Kh) / stride + 1)
    """
    import torch.nn as nn

    # Forward hook to capture spatial dims
    hooks, spatial = [], {}

    def make_hook(name):
        def hook(module, inp, out):
            if hasattr(out, "shape"):
                spatial[name] = tuple(out.shape)
        return hook

    handles = []
    for name, mod in model.named_modules():
        handles.append(mod.register_forward_hook(make_hook(name)))

    x = torch.randn(*INPUT_SHAPE)
    with torch.no_grad():
        model(x)
    for h in handles:
        h.remove()

    total_flops = 0
    layer_breakdown = {}

    for name, mod in model.named_modules():
        if isinstance(mod, nn.Conv2d):
            shape = spatial.get(name)
            if shape is None:
                continue
            _, cout, hout, wout = shape
            cin  = mod.in_channels
            kh   = mod.kernel_size[0] if isinstance(mod.kernel_size, tuple) else mod.kernel_size
            kw   = mod.kernel_size[1] if isinstance(mod.kernel_size, tuple) else mod.kernel_size
            # MACs = Cout × (Cin/groups) × Kh × Kw × Hout × Wout
            macs = cout * (cin // mod.groups) * kh * kw * hout * wout
            flops = 2 * macs
            total_flops += flops
            layer_breakdown[name] = {"type": "Conv2d", "flops": flops, "macs": macs,
                                      "shape_out": list(shape)}

        elif isinstance(mod, nn.Linear):
            flops = 2 * mod.in_features * mod.out_features
            total_flops += flops
            layer_breakdown[name] = {"type": "Linear", "flops": flops,
                                      "in": mod.in_features, "out": mod.out_features}

        elif isinstance(mod, nn.BatchNorm2d):
            # BatchNorm FLOPs are deliberately excluded here.
            # They are negligible vs Conv (typically <0.1% of total) and
            # most standard tools (fvcore, torchprofile, thop) omit them.
            # If fused into Conv by TensorRT/ORT, they contribute zero extra ops.
            # Including them with an ad-hoc formula (4×C×H×W) is non-standard
            # and inflates the reported count without improving comparability.
            pass

    return {
        "total_flops"    : total_flops,
        "gflops"         : round(total_flops / 1e9, 4),
        "layer_breakdown": layer_breakdown,
        "formula": {
            "conv2d" : "FLOPs = 2 × (Cin/groups) × Cout × Kh × Kw × Hout × Wout",
            "linear" : "FLOPs = 2 × in_features × out_features",
            "batchnorm": "excluded — negligible vs Conv, fused by TRT/ORT, omitted by fvcore/thop",
        },
        "note": (
            "FLOPs are IDENTICAL for INT8 and FP32 models — quantization does not "
            "change the op count. INT8 throughput advantage is a hardware property "
            "(wider SIMD lanes), not a reduction in arithmetic operations."
        ),
    }

def measure_gpu_throughput_ms(model, device, n: int = INFERENCE_RUNS) -> Dict:
    """
    GPU latency using CUDA events (more accurate than time.perf_counter).
    CUDA events are synchronized on the GPU timeline; avoids CPU-GPU sync overhead.

    Returns mean, std, min, max in milliseconds.
    """
    model = model.to(device).eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32, device=device)

    # Warmup
    with torch.no_grad():
        for _ in range(10):
            model(dummy)
    torch.cuda.synchronize(device)

    timings = []
    start_event = torch.cuda.Event(enable_timing=True)
    end_event   = torch.cuda.Event(enable_timing=True)

    with torch.no_grad():
        for _ in range(n):
            start_event.record()
            model(dummy)
            end_event.record()
            torch.cuda.synchronize()
            timings.append(start_event.elapsed_time(end_event))

    arr = np.array(timings)
    return {
        "mean_ms"  : round(float(arr.mean()), 4),
        "std_ms"   : round(float(arr.std()),  4),
        "min_ms"   : round(float(arr.min()),  4),
        "max_ms"   : round(float(arr.max()),  4),
        "samples"  : n,
        "batch_size": BATCH_SIZE,
        "throughput_imgs_per_sec": round(BATCH_SIZE / (float(arr.mean()) / 1000), 1),
    }

def measure_cpu_ms(model, n: int = INFERENCE_RUNS) -> float:
    model = model.cpu().eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32)
    with torch.no_grad():
        for _ in range(5):
            model(dummy)
        t0 = time.perf_counter()
        for _ in range(n):
            model(dummy)
    return round((time.perf_counter() - t0) / n * 1000, 4)

def disk_size_mb(obj) -> float:
    """Works for PyTorch models and ONNX files (pass path as str)."""
    if isinstance(obj, str):
        return os.path.getsize(obj) / 1024 ** 2
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(obj, tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)


# ══════════════════════════════════════════════════════════════════════════════
# Evaluation
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def evaluate_torch(model, loader, device) -> Dict:
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    model = model.to(device).eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(device)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def evaluate_onnx(session, loader) -> Dict:
    """Run inference with ONNX Runtime session."""
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    input_name = session.get_inputs()[0].name
    preds, labels = [], []
    for inputs, lbls in loader:
        out = session.run(None, {input_name: inputs.numpy()})[0]
        preds.extend(np.argmax(out, axis=1))
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


# ══════════════════════════════════════════════════════════════════════════════
# ONNX Export
# ══════════════════════════════════════════════════════════════════════════════
def export_to_onnx(model, output_path: str, opset: int = 17) -> str:
    """
    Export PyTorch FP32 model to ONNX.
    opset 17+ is required for INT8 quantization ops (QLinearConv, QLinearMatMul).
    Dynamic axes allow variable batch size at inference time.
    """
    dummy = torch.randn(*INPUT_SHAPE)
    torch.onnx.export(
        model.cpu().eval(),
        dummy,
        output_path,
        opset_version        = opset,
        input_names          = ["input"],
        output_names         = ["output"],
        dynamic_axes         = {"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        do_constant_folding  = True,
        export_params        = True,
    )
    print(f"        ✓ ONNX exported → {output_path}  "
          f"({os.path.getsize(output_path)/1024**2:.2f} MB, opset={opset})")
    return output_path


# ══════════════════════════════════════════════════════════════════════════════
# 1. ONNX Runtime — INT8 Static Quantization (GPU inference)
# ══════════════════════════════════════════════════════════════════════════════
"""
ONNX Runtime quantization workflow:
  1. Export FP32 model to ONNX (float32 weights + ops)
  2. Insert QuantizeLinear / DequantizeLinear nodes around each op
  3. Run calibration data through — ORT records per-tensor statistics
  4. Compute S = (x_max - x_min) / 255 and Z = round(-x_min / S) per tensor
  5. Freeze S and Z into the graph → INT8 model
  6. At inference: CUDAExecutionProvider dispatches each quantized op to the
     fastest available GPU kernel for that op type:
       QLinearConv   → cuDNN INT8 (when supported) or custom CUDA kernel
       QLinearMatMul → cuBLASLt INT8 GEMM
       QLinearAdd    → custom elementwise CUDA kernel
     Kernel selection depends on opset, graph structure, and cuDNN version.
     Not every QLinear op is guaranteed to use cuDNN — this is determined
     at session creation by ORT's graph partitioner.

Calibration strategies in ORT:
  MinMax      — exact observed range; sensitive to outliers → S can be large
  Entropy     — histogram-based KL minimization; better for long-tail activations
  Percentile  — clips at [p, 1-p] to remove outliers; reduces S → finer grid
"""
def run_onnx_ptq(fp32_model, test_loader, calib_loader,
                 baseline_acc: float, base_disk: float,
                 device: str, flops_fp32: Dict) -> List[Dict]:
    import onnxruntime
    from onnxruntime.quantization import (
        quantize_static, CalibrationDataReader,
        QuantType, QuantFormat, CalibrationMethod
    )

    print("\n  [1/3] ONNX Runtime — Static INT8 PTQ")
    print("        Framework: onnxruntime-gpu (CUDAExecutionProvider)")
    print("        Kernels:   cuDNN INT8 (Conv), cuBLASLt (MatMul), custom (Add)")
    print("        S/Z:       calibrated once from calib data; frozen in graph")

    # Export FP32 → ONNX
    fp32_onnx = "resnet50_fp32.onnx"
    export_to_onnx(fp32_model, fp32_onnx)

    # Calibration data reader
    class CIFARCalibReader(CalibrationDataReader):
        """
        Feeds CIFAR-10 images to the ONNX calibrator.
        Observer collects x_min / x_max per tensor.
        S = (x_max - x_min) / 255,  Z = round(-x_min / S)

        Correctness notes:
        - imgs.numpy() produces float32 (CIFAR normalized by transforms.Normalize).
        - np.ascontiguousarray ensures the buffer is C-contiguous, which ORT requires.
        - No additional normalization is applied here because the DataLoader's
          transforms.Normalize already applied CIFAR mean/std subtraction.
        """
        def __init__(self, loader, max_batches=CALIB_BATCHES):
            self.data    = iter(loader)
            self.batches = 0
            self.max     = max_batches

        def get_next(self):
            if self.batches >= self.max:
                return None
            try:
                imgs, _ = next(self.data)
                self.batches += 1
                # Ensure contiguous float32 — ORT calibrator requirement
                arr = np.ascontiguousarray(imgs.numpy(), dtype=np.float32)
                return {"input": arr}
            except StopIteration:
                return None

    rows = []
    calibration_methods = {
        "MinMax"     : CalibrationMethod.MinMax,
        "Entropy"    : CalibrationMethod.Entropy,
        "Percentile" : CalibrationMethod.Percentile,
    }

    for calib_name, calib_method in calibration_methods.items():
        print(f"        Calibration: {calib_name:12s}", end="", flush=True)
        int8_onnx = f"resnet50_int8_{calib_name.lower()}.onnx"
        try:
            # Quantize: inserts QuantizeLinear + DequantizeLinear nodes
            quantize_static(
                model_input          = fp32_onnx,
                model_output         = int8_onnx,
                calibration_data_reader = CIFARCalibReader(calib_loader),
                quant_format         = QuantFormat.QDQ,       # QuantizeDequantize pairs
                per_channel          = True,                   # per-channel weights
                reduce_range         = False,
                weight_type          = QuantType.QInt8,
                activation_type      = QuantType.QUInt8,
                calibrate_method     = calib_method,
                extra_options        = {
                    "EnableSubgraph"              : True,
                    "MatMulConstBOnly"            : False,
                    "AddQDQPairToWeight"          : True,
                    "QuantizeAllFixedZeroPointTensors": True,
                },
            )

            # Inference session — prefer CUDA, fall back to CPU
            providers = (["CUDAExecutionProvider", "CPUExecutionProvider"]
                         if "CUDAExecutionProvider" in onnxruntime.get_available_providers()
                         else ["CPUExecutionProvider"])
            sess_opts = onnxruntime.SessionOptions()
            sess_opts.graph_optimization_level = (
                onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL
            )
            session = onnxruntime.InferenceSession(int8_onnx, sess_opts,
                                                    providers=providers)
            active_provider = session.get_providers()[0]

            metrics  = evaluate_onnx(session, test_loader)
            disk_mb  = disk_size_mb(int8_onnx)

            # ORT latency (wall-clock; for true GPU use CUDA events in TRT section)
            dummy_np = np.random.randn(BATCH_SIZE, 3, 32, 32).astype(np.float32)
            input_name = session.get_inputs()[0].name
            with torch.no_grad():
                for _ in range(5):
                    session.run(None, {input_name: dummy_np})
            t0 = time.perf_counter()
            for _ in range(INFERENCE_RUNS):
                session.run(None, {input_name: dummy_np})
            latency_ms = (time.perf_counter() - t0) / INFERENCE_RUNS * 1000

            # FLOPs are identical between INT8 and FP32 — quantization changes
            # the datatype, not the operation count. Report the same FP32 FLOPs
            # and use measured latency to characterise the actual GPU speedup.
            flops_int8 = {
                "total_flops"  : flops_fp32.get("total_flops"),
                "gflops"       : flops_fp32.get("gflops_fp32") or flops_fp32.get("gflops"),
                "note": (
                    "FLOPs are IDENTICAL for INT8 and FP32 — quantization does not "
                    "reduce op count. GPU speedup comes from hardware-level INT8 SIMD "
                    "width (~4× Ampere Tensor Core throughput) and reduced memory "
                    "bandwidth (~4× fewer bytes per weight). "
                    "Use inference_latency.mean_ms for real-world speedup comparison."
                ),
            }

            row = {
                "backend"            : "onnxruntime_int8",
                "calibration_method" : calib_name,
                "execution_provider" : active_provider,
                "description"        : (
                    f"ONNX Runtime static INT8 PTQ (calibration={calib_name}). "
                    "QDQ format: QuantizeLinear/DequantizeLinear pairs wrap each op. "
                    "S = (x_max − x_min) / (2^8 − 1),  Z = round(−x_min / S). "
                    f"Calibrated on {CALIB_SIZE} CIFAR-10 training images. "
                    "GPU inference via CUDAExecutionProvider: cuDNN INT8 for Conv, "
                    "cuBLASLt INT8 GEMM for MatMul; actual kernel depends on "
                    "op type, opset, and ORT graph partitioner at session creation."
                ),
                "quantization_math"  : {
                    "S"       : "S = (x_max − x_min) / (2^8 − 1)   [255 levels for UINT8 activations]",
                    "Z"       : "Z = round(−x_min / S)",
                    "Q(x)"    : "Q(x) = clip(round(x / S) + Z, 0, 255)  → stored as UINT8",
                    "x̂"       : "x̂ = S · (Q(x) − Z)  → reconstructed at op boundary",
                    "ε_bound" : "|ε| ≤ S/2  (smaller S ↔ finer quantisation grid ↔ less error)",
                    "format"  : "QDQ (QuantizeLinear / DequantizeLinear node pairs, ONNX opset 10+)",
                },
                "ops_quantized"      : "Conv, MatMul, Gemm, Add (residual) — subject to graph partitioner",
                "weight_dtype"       : "INT8 per-channel symmetric",
                "activation_dtype"   : "UINT8 per-tensor affine",
                "calibration_samples": CALIB_SIZE,
                "accuracy"           : round(metrics["accuracy"],  6),
                "precision"          : round(metrics["precision"], 6),
                "recall"             : round(metrics["recall"],    6),
                "f1"                 : round(metrics["f1"],        6),
                "accuracy_drop"      : round(baseline_acc - metrics["accuracy"], 6),
                "size_disk_mb"       : round(disk_mb, 4),
                "disk_saved_mb"      : round(base_disk - disk_mb, 4),
                "compression_ratio"  : round(base_disk / disk_mb, 4) if disk_mb else None,
                "inference_latency"  : {"mean_ms": round(latency_ms, 4),
                                         "batch_size": BATCH_SIZE},
                "flops"              : flops_int8,
            }
            print(f" → Acc: {metrics['accuracy']:.4f}  "
                  f"Drop: {baseline_acc - metrics['accuracy']:+.4f}  "
                  f"Disk: {disk_mb:.2f} MB  Lat: {latency_ms:.1f} ms ({active_provider})")
            rows.append(row)

        except Exception as e:
            print(f" → FAILED: {e}")
            rows.append({
                "backend": "onnxruntime_int8",
                "calibration_method": calib_name,
                "description": f"FAILED: {e}",
                "accuracy": None,
            })

    # Cleanup exported ONNX files
    for f in [fp32_onnx] + [f"resnet50_int8_{m.lower()}.onnx"
                              for m in calibration_methods]:
        if os.path.exists(f):
            os.remove(f)

    return rows


# ══════════════════════════════════════════════════════════════════════════════
# 2. TensorRT — INT8 PTQ with Calibration Cache
# ══════════════════════════════════════════════════════════════════════════════
"""
TensorRT INT8 PTQ workflow:
  1. Parse ONNX model into TRT NetworkDefinition
  2. Set INT8 mode + attach IInt8EntropyCalibrator2
  3. Build engine: TRT fuses layers, selects fastest INT8 kernels per GPU
  4. Serialize engine to disk (.trt) — device-specific binary!

TensorRT advantages over ORT INT8:
  - Kernel autotuning at build time (profile-guided layer selection)
  - Layer fusion: Conv + BN + ReLU → single kernel call, fewer memory reads
  - Memory layout optimization (NHWC channel-last for Tensor Core alignment)
  - ~2-5× faster than ORT GPU on A100/H100 for ResNet-class models

IInt8EntropyCalibrator2 — how it actually works:
  TensorRT does NOT solve a closed-form argmin KL problem at runtime.
  The real procedure is:
    1. Collect activation histograms over calibration batches (128 bins by default)
    2. For each candidate saturation threshold T in {max/128, 2*max/128, ...}:
         - Quantize the histogram to 128 INT8 bins using that threshold
         - Compute KL divergence between original and quantized histogram
    3. Select the T* with minimum KL divergence
    4. Set S = T* / 127  (symmetric INT8 range [-127, 127])
  This is a heuristic histogram search, not an analytic argmin.
  The KL minimization framing (S* = argmin KL) is conceptually correct as
  motivation but is an approximation of what TRT actually computes.

  Why IInt8EntropyCalibrator2 over MinMax?
  ResNet activations post-ReLU have long right tails. MinMax sets
  S = max_activation / 127, which wastes precision on rare large values.
  Entropy calibration clips these outliers at T*, concentrating quantization
  levels where activation density is highest → lower average ε.
"""
def run_tensorrt_ptq(fp32_model, test_loader, calib_loader,
                     baseline_acc: float, base_disk: float,
                     device: str, flops_fp32: Dict) -> Dict:
    print("\n  [2/3] TensorRT — INT8 PTQ")
    try:
        import tensorrt as trt
        import pycuda.driver as cuda
        import pycuda.autoinit
    except ImportError as e:
        print(f"        ⚠ TensorRT/pycuda not available: {e}")
        print("          Install: pip install tensorrt pycuda")
        return {
            "backend"    : "tensorrt_int8",
            "description": f"SKIPPED — TensorRT not installed: {e}",
            "accuracy"   : None,
            "install"    : "pip install tensorrt pycuda",
        }

    print("        Framework: TensorRT INT8 engine")
    print("        Fusion:    Conv-BN-ReLU automatic, memory layout optimized")
    print("        Calib:     IInt8EntropyCalibrator2 (KL-divergence minimization)")

    TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
    fp32_onnx  = "resnet50_trt_fp32.onnx"

    try:
        export_to_onnx(fp32_model, fp32_onnx)

        # ── Calibrator ─────────────────────────────────────────────────────────
        class CIFARCalibrator(trt.IInt8EntropyCalibrator2):
            """
            Feeds calibration batches to TensorRT.
            TRT uses these to compute per-tensor activation statistics,
            then solves: S* = argmin KL(P_fp32 || P_int8(·, S))
            """
            def __init__(self, loader, cache_file="trt_calib.cache"):
                super().__init__()
                self.loader     = iter(loader)
                self.batches    = 0
                self.cache_file = cache_file
                self.device_mem = None
                batch, _ = next(iter(loader))
                self.batch_size = batch.shape[0]
                self.buf_size   = int(np.prod(batch.shape) * 4)  # float32 bytes
                self.device_mem = cuda.mem_alloc(self.buf_size)

            def get_batch_size(self):
                return self.batch_size

            def get_batch(self, names):
                if self.batches >= CALIB_BATCHES:
                    return None
                try:
                    imgs, _ = next(self.loader)
                    cuda.memcpy_htod(self.device_mem, imgs.numpy().astype(np.float32))
                    self.batches += 1
                    return [int(self.device_mem)]
                except StopIteration:
                    return None

            def read_calibration_cache(self):
                if os.path.exists(self.cache_file):
                    with open(self.cache_file, "rb") as f:
                        return f.read()
                return None

            def write_calibration_cache(self, cache):
                with open(self.cache_file, "wb") as f:
                    f.write(cache)

        # ── Build engine ───────────────────────────────────────────────────────
        builder  = trt.Builder(TRT_LOGGER)
        network  = builder.create_network(
            1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
        )
        parser   = trt.OnnxParser(network, TRT_LOGGER)
        with open(fp32_onnx, "rb") as f:
            if not parser.parse(f.read()):
                raise RuntimeError(f"ONNX parse error: {parser.get_error(0)}")

        config = builder.create_builder_config()
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)  # 1 GB
        config.set_flag(trt.BuilderFlag.INT8)
        config.set_flag(trt.BuilderFlag.FP16)   # allow FP16 for unsupported INT8 ops

        calibrator = CIFARCalibrator(calib_loader)
        config.int8_calibrator = calibrator

        # Profile for dynamic batch
        profile = builder.create_optimization_profile()
        profile.set_shape("input",
                           min=(1, 3, 32, 32),
                           opt=(BATCH_SIZE, 3, 32, 32),
                           max=(BATCH_SIZE * 2, 3, 32, 32))
        config.add_optimization_profile(profile)

        print("        Building TRT INT8 engine (this may take several minutes)...")
        t_build = time.perf_counter()
        serialized = builder.build_serialized_network(network, config)
        build_time = time.perf_counter() - t_build
        print(f"        Engine built in {build_time:.1f}s")

        trt_path = "resnet50_int8.trt"
        with open(trt_path, "wb") as f:
            f.write(serialized)

        # ── Inference ─────────────────────────────────────────────────────────
        runtime = trt.Runtime(TRT_LOGGER)
        engine  = runtime.deserialize_cuda_engine(serialized)
        context = engine.create_execution_context()
        context.set_input_shape("input", (BATCH_SIZE, 3, 32, 32))

        # Allocate GPU buffers
        d_input  = cuda.mem_alloc(BATCH_SIZE * 3 * 32 * 32 * 4)
        d_output = cuda.mem_alloc(BATCH_SIZE * NUM_CLASSES * 4)
        stream   = cuda.Stream()

        def infer_trt(x_np):
            cuda.memcpy_htod_async(d_input, x_np.astype(np.float32), stream)
            context.execute_async_v2(
                bindings=[int(d_input), int(d_output)], stream_handle=stream.handle
            )
            out = np.empty((BATCH_SIZE, NUM_CLASSES), dtype=np.float32)
            cuda.memcpy_dtoh_async(out, d_output, stream)
            stream.synchronize()
            return out

        # Evaluate
        from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
        preds, labels = [], []
        for inputs, lbls in test_loader:
            out = infer_trt(inputs.numpy())
            preds.extend(np.argmax(out, 1))
            labels.extend(lbls.numpy())
        y_pred, y_true = np.array(preds), np.array(labels)
        metrics = {
            "accuracy" : float(accuracy_score(y_true, y_pred)),
            "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
            "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
            "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        }

        # GPU latency with CUDA events
        dummy_np = np.random.randn(BATCH_SIZE, 3, 32, 32).astype(np.float32)
        start_ev = cuda.Event()
        end_ev   = cuda.Event()
        for _ in range(5):
            infer_trt(dummy_np)
        timings = []
        for _ in range(INFERENCE_RUNS):
            start_ev.record()
            infer_trt(dummy_np)
            end_ev.record()
            end_ev.synchronize()
            timings.append(start_ev.time_till(end_ev))
        arr = np.array(timings)

        disk_mb = disk_size_mb(trt_path)

        row = {
            "backend"          : "tensorrt_int8",
            "description"      : (
                "TensorRT INT8 engine: ONNX → TRT with IInt8EntropyCalibrator2. "
                "Layer fusion (Conv+BN+ReLU), memory layout optimization (NHWC), "
                "kernel autotuning at build time for this specific GPU. "
                "Calibration: histogram-based KL-divergence threshold search "
                "(heuristic, not closed-form argmin). "
                f"Engine built in {build_time:.1f}s; NOT portable across GPU architectures."
            ),
            "quantization_math": {
                "calibration_procedure": (
                    "Collect per-tensor activation histogram over calib batches. "
                    "For each candidate threshold T: quantize histogram to 128 bins, "
                    "compute KL(original_hist || quantized_hist). "
                    "Select T* = argmin KL over histogram candidates. "
                    "Set S = T* / 127  (symmetric INT8 in [-127, 127])."
                ),
                "S_formula"    : "S = T* / 127  where T* minimizes histogram KL divergence",
                "note"         : "KL argmin is a heuristic search over histogram bins, not analytic",
                "dtype_weights": "INT8 per-channel symmetric",
                "dtype_acts"   : "INT8 per-tensor (calibrated threshold)",
                "fusion"       : "Conv+BN+ReLU fused to single kernel; residual add absorbed",
            },
            "build_time_sec"    : round(build_time, 2),
            "calibration_method": "IInt8EntropyCalibrator2 (histogram KL threshold search)",
            "calibration_samples": CALIB_SIZE,
            "accuracy"         : round(metrics["accuracy"],  6),
            "precision"        : round(metrics["precision"], 6),
            "recall"           : round(metrics["recall"],    6),
            "f1"               : round(metrics["f1"],        6),
            "accuracy_drop"    : round(baseline_acc - metrics["accuracy"], 6),
            "size_disk_mb"     : round(disk_mb, 4),
            "disk_saved_mb"    : round(base_disk - disk_mb, 4),
            "compression_ratio": round(base_disk / disk_mb, 4),
            "inference_latency": {
                "mean_ms"             : round(float(arr.mean()), 4),
                "std_ms"              : round(float(arr.std()),  4),
                "min_ms"              : round(float(arr.min()),  4),
                "max_ms"              : round(float(arr.max()),  4),
                "samples"             : INFERENCE_RUNS,
                "batch_size"          : BATCH_SIZE,
                "throughput_imgs_sec" : round(BATCH_SIZE / (float(arr.mean()) / 1000), 1),
                "timing_method"       : "CUDA events (GPU-synchronized, not wall-clock)",
            },
            "flops": {
                "total_flops"  : flops_fp32.get("total_flops"),
                "gflops"       : flops_fp32.get("gflops_fp32") or flops_fp32.get("gflops"),
                "note": (
                    "FLOPs are IDENTICAL to FP32 baseline — quantization does not "
                    "reduce op count. TRT speedup comes from: (1) INT8 Tensor Core "
                    "throughput (~4× Ampere), (2) Conv+BN+ReLU kernel fusion reducing "
                    "memory round-trips, (3) NHWC layout for Tensor Core alignment. "
                    "Use inference_latency.mean_ms for actual speedup."
                ),
            },
        }
        print(f"        → Acc: {metrics['accuracy']:.4f}  "
              f"Drop: {baseline_acc - metrics['accuracy']:+.4f}  "
              f"Disk: {disk_mb:.2f} MB  GPU: {float(arr.mean()):.1f} ms")

    except Exception as e:
        print(f"        → FAILED: {e}")
        row = {
            "backend"    : "tensorrt_int8",
            "description": f"FAILED: {e}",
            "accuracy"   : None,
        }
    finally:
        for f in [fp32_onnx, "trt_calib.cache"]:
            if os.path.exists(f):
                os.remove(f)

    return row


# ══════════════════════════════════════════════════════════════════════════════
# 3. PyTorch CUDA — Dynamic PTQ + PT2E Inductor INT8
# ══════════════════════════════════════════════════════════════════════════════
"""
PyTorch GPU quantization options:
  a) torch.quantization.quantize_dynamic — Linear → qint8 storage
     Works on CUDA tensors. However: PyTorch dynamic quantization was designed
     for CPU inference. On CUDA, the quantized weights are stored as INT8 but
     dequantized to FP32/FP16 before the CUDA GEMM kernel executes. This means:
       - Memory: ~2× reduction (INT8 storage vs FP32)
       - Compute: NO INT8 GEMM savings — the actual kernel runs in FP16/FP32
     This is NOT the same as true INT8 GEMM. Conv2d stays FP32 regardless.
     GPU behavior is not guaranteed to match CPU dynamic quant semantics.

  b) PT2E (PyTorch 2 Export) + X86InductorQuantizer → torch.compile
     Naming note: X86InductorQuantizer is named for its x86 CPU targeting history
     but it also annotates GPU-eligible ops when compiled with the Inductor backend.
     The actual GPU kernel (Triton or cuBLAS INT8) is selected by Inductor at
     compile time based on the target device, NOT by the quantizer name.
     torch.compile(backend='inductor') generates INT8 Triton kernels for CUDA.
     Current status (torch 2.3–2.5): Conv2d + Linear INT8 on CUDA is supported
     but maturing. Some ops may fall back to FP16 depending on shape/dtype support.
     Report measured latency; do not assume fully INT8 execution.
  torch < 2.4 : torch.ao.quantization.quantize_pt2e
                torch.ao.quantization.quantizer.x86_inductor_quantizer
                torch._export.capture_pre_autograd_graph
  torch >= 2.4: torch.ao.quantization.quantize_pt2e  (same)
                torch.ao.quantization.quantizer.xnnpack_quantizer (CPU)
                torch.export.export  (replaces capture_pre_autograd_graph)
  torch >= 2.5: torch.quantization.quantize_pt2e  (re-exported alias)

This function resolves all three variants at runtime.
"""

def _resolve_pt2e_imports():
    """
    Resolve PT2E symbols across PyTorch 2.3 / 2.4 / 2.5+ import paths.
    Returns (prepare_pt2e, convert_pt2e, Quantizer, get_config, export_fn)
    or raises ImportError with a clear message listing what was tried.

    Canonical locations by version
    ───────────────────────────────
    torch 2.3–2.4
      prepare/convert : torch.ao.quantization.quantize_pt2e
      quantizer       : torch.ao.quantization.quantizer.x86_inductor_quantizer
      export          : torch._export.capture_pre_autograd_graph
                        (deprecated in 2.4, removed in 2.5)

    torch 2.5+
      prepare/convert : torch.ao.quantization.quantize_pt2e  (still valid)
      quantizer       : torch.ao.quantization.quantizer.x86_inductor_quantizer
      export          : torch.export.export  (stable public API)
    """
    errors = []

    # ── Step 1: prepare_pt2e / convert_pt2e ───────────────────────────────────
    prepare_pt2e = convert_pt2e = None
    for mod_path in [
        "torch.ao.quantization.quantize_pt2e",   # 2.3–2.5+
        "torch.quantization.quantize_pt2e",       # re-export alias (some builds)
    ]:
        try:
            import importlib
            m = importlib.import_module(mod_path)
            prepare_pt2e = m.prepare_pt2e
            convert_pt2e = m.convert_pt2e
            break
        except (ImportError, AttributeError) as e:
            errors.append(f"  {mod_path}: {e}")

    if prepare_pt2e is None:
        raise ImportError(
            "Cannot find prepare_pt2e / convert_pt2e. Tried:\n" + "\n".join(errors) +
            "\nRequires PyTorch >= 2.3.  Install: pip install torch>=2.3"
        )

    # ── Step 2: X86InductorQuantizer ──────────────────────────────────────────
    Quantizer = get_config = None
    for mod_path, cls, cfg in [
        (
            "torch.ao.quantization.quantizer.x86_inductor_quantizer",
            "X86InductorQuantizer",
            "get_default_x86_inductor_quantization_config",
        ),
    ]:
        try:
            import importlib
            m = importlib.import_module(mod_path)
            Quantizer  = getattr(m, cls)
            get_config = getattr(m, cfg)
            break
        except (ImportError, AttributeError) as e:
            errors.append(f"  {mod_path}.{cls}: {e}")

    if Quantizer is None:
        raise ImportError(
            "Cannot find X86InductorQuantizer. Tried:\n" + "\n".join(errors) +
            "\nRequires PyTorch >= 2.3 built with Inductor."
        )

    # ── Step 3: graph export function ─────────────────────────────────────────
    # torch.export.export is the stable public API (2.1+).
    # capture_pre_autograd_graph was the older internal path (removed in 2.5).
    export_fn = None

    # Preferred: torch.export.export (wraps model + example_inputs)
    try:
        from torch.export import export as _torch_export

        def export_fn(model, example_args):
            # torch.export.export returns an ExportedProgram; PT2E needs the
            # .module() graph module for prepare_pt2e.
            ep = _torch_export(model, example_args)
            return ep.module()

    except (ImportError, AttributeError) as e:
        errors.append(f"  torch.export.export: {e}")

    # Fallback: capture_pre_autograd_graph (torch 2.3–2.4)
    if export_fn is None:
        try:
            from torch._export import capture_pre_autograd_graph as _cap

            def export_fn(model, example_args):
                return _cap(model, example_args)

        except (ImportError, AttributeError) as e:
            errors.append(f"  torch._export.capture_pre_autograd_graph: {e}")

    if export_fn is None:
        raise ImportError(
            "Cannot find a graph export function. Tried:\n" + "\n".join(errors)
        )

    return prepare_pt2e, convert_pt2e, Quantizer, get_config, export_fn


def run_torch_cuda_ptq(fp32_model, test_loader, calib_loader,
                        baseline_acc: float, base_disk: float,
                        device: str, flops_fp32: Dict) -> List[Dict]:
    print("\n  [3/3] PyTorch CUDA — Dynamic PTQ + PT2E Inductor INT8")
    rows = []
    import torch.nn as nn

    # ── 3a. Dynamic PTQ on CUDA ────────────────────────────────────────────────
    print("        3a. Dynamic PTQ (Linear → qint8) on CUDA")
    try:
        q_dyn = torch.quantization.quantize_dynamic(
            copy.deepcopy(fp32_model), {nn.Linear}, dtype=torch.qint8
        )
        # NOTE: Dynamic PTQ model keeps FP32 compute; weights are INT8 in memory.
        # CUDA does: dequant(INT8 weight) → FP16 → cuBLAS GEMM
        # True INT8 GEMM requires static calibration or TensorRT / PT2E.
        q_dyn = q_dyn.to(device).eval()

        metrics = evaluate_torch(q_dyn, test_loader, device)
        lat     = measure_gpu_throughput_ms(q_dyn, device)
        disk_mb = disk_size_mb(q_dyn.cpu())

        rows.append({
            "backend"         : "torch_cuda_dynamic",
            "description"     : (
                "PyTorch dynamic PTQ on CUDA: Linear weights stored as qint8 in memory. "
                "IMPORTANT: on CUDA, weights are dequantized to FP32/FP16 BEFORE the "
                "cuBLAS GEMM kernel executes — this is NOT true INT8 GEMM. "
                "Benefit is memory bandwidth reduction (~2× for Linear weights), "
                "not compute savings. Conv2d stays FP32. No calibration needed. "
                "GPU dynamic quant behavior is not identical to CPU semantics."
            ),
            "quantization_math": {
                "scope"       : "Linear weights only (Conv2d stays FP32)",
                "storage"     : "qint8 (8-bit signed, ~2× memory reduction vs FP32)",
                "compute"     : "FP16/FP32 after weight dequant — NOT INT8 GEMM on GPU",
                "S"           : "computed per-tensor from weight min/max at runtime",
                "gpu_reality" : (
                    "CUDA path: dequant(INT8 weight) → FP16 → cuBLAS FP16 GEMM. "
                    "Savings are bandwidth-bound, not compute-bound."
                ),
            },
            "ops_quantized"   : "Linear weights only (Conv2d stays FP32)",
            "calibration_samples": 0,
            "accuracy"        : round(metrics["accuracy"],  6),
            "precision"       : round(metrics["precision"], 6),
            "recall"          : round(metrics["recall"],    6),
            "f1"              : round(metrics["f1"],        6),
            "accuracy_drop"   : round(baseline_acc - metrics["accuracy"], 6),
            "size_disk_mb"    : round(disk_mb, 4),
            "disk_saved_mb"   : round(base_disk - disk_mb, 4),
            "compression_ratio": round(base_disk / disk_mb, 4) if disk_mb else None,
            "inference_latency": lat,
            "flops": {
                "total_flops"  : flops_fp32.get("total_flops"),
                "gflops"       : flops_fp32.get("gflops_fp32") or flops_fp32.get("gflops"),
                "note": (
                    "FLOPs identical to FP32 baseline — dynamic PTQ does not reduce op count. "
                    "GPU compute runs in FP16/FP32 after weight dequant; only memory "
                    "bandwidth is reduced (~2× for Linear layers)."
                ),
            },
        })
        print(f"           → Acc: {metrics['accuracy']:.4f}  "
              f"GPU: {lat['mean_ms']:.1f} ms  Disk: {disk_mb:.2f} MB")
    except Exception as e:
        print(f"           → FAILED: {e}")
        rows.append({"backend": "torch_cuda_dynamic",
                     "description": f"FAILED: {e}", "accuracy": None})

    # ── 3b. PT2E + X86InductorQuantizer → torch.compile ──────────────────────
    print("        3b. PT2E (torch.export + X86InductorQuantizer) → torch.compile")
    try:
        # Resolve imports across PyTorch 2.3 / 2.4 / 2.5+
        prepare_pt2e, convert_pt2e, Quantizer, get_config, export_fn = (
            _resolve_pt2e_imports()
        )
        torch_ver = tuple(int(x) for x in torch.__version__.split(".")[:2])
        print(f"           PT2E imports resolved  "
              f"(torch {torch.__version__})")

        # Export ATen graph — works for both torch.export.export (2.5+) and
        # capture_pre_autograd_graph (2.3–2.4)
        example_args = (torch.randn(1, 3, 32, 32, device=device),)
        m = copy.deepcopy(fp32_model).to(device).eval()
        exported = export_fn(m, example_args)

        # Annotate + prepare observers
        quantizer = Quantizer()
        quantizer.set_global(get_config())
        prepared = prepare_pt2e(exported, quantizer)

        # Calibrate: run calib data so observers collect x_min, x_max
        # S = (x_max − x_min) / 255,  Z = round(−x_min / S)
        prepared.eval()
        with torch.no_grad():
            for i, (imgs, _) in enumerate(calib_loader):
                prepared(imgs.to(device))
                if i + 1 >= CALIB_BATCHES:
                    break

        # Freeze S/Z → INT8 ops (QLinearConv / QLinearMatMul in Inductor IR)
        q_pt2e = convert_pt2e(prepared)

        # Compile: Inductor lowers to INT8 Triton or cuBLAS kernels
        q_compiled = torch.compile(q_pt2e, mode="max-autotune",
                                    backend="inductor")

        metrics = evaluate_torch(q_compiled, test_loader, device)
        lat     = measure_gpu_throughput_ms(q_compiled, device)

        rows.append({
            "backend"         : "torch_pt2e_inductor_int8",
            "torch_version"   : torch.__version__,
            "description"     : (
                "PT2E (PyTorch 2 Export) + X86InductorQuantizer + torch.compile: "
                "torch.export / capture_pre_autograd_graph → "
                "prepare_pt2e (inserts MinMaxObservers) → calibrate (collects x_min/x_max) → "
                "convert_pt2e (freezes S/Z as constants) → "
                "torch.compile(mode='max-autotune', backend='inductor'). "
                "Naming note: X86InductorQuantizer annotates ops for Inductor lowering; "
                "the actual GPU kernels (Triton INT8 or cuBLAS INT8) are selected by "
                "Inductor at compile time based on device — the quantizer name is "
                "historical and does not imply CPU-only execution. "
                "MATURITY WARNING (torch 2.3–2.5): CUDA INT8 via PT2E is still evolving; "
                "some ops may fall back to FP16 mixed execution. "
                "Measure actual latency rather than assuming full INT8 throughput."
            ),
            "quantization_math": {
                "S"           : "S = (x_max − x_min) / (2^8 − 1)  [from MinMaxObserver over calib data]",
                "Z"           : "Z = round(−x_min / S)",
                "Q(x)"        : "Q(x) = clip(round(x / S) + Z, 0, 255)  → UINT8",
                "compute"     : "Inductor selects INT8 Triton kernel or cuBLAS INT8 per op at compile time",
                "fusion"      : "Inductor may fuse quant+conv+dequant into single kernel",
                "dtype_w"     : "INT8 symmetric per-channel",
                "dtype_act"   : "UINT8 per-tensor affine",
                "cuda_caveat" : (
                    "Not all ops are guaranteed to execute as INT8 on CUDA. "
                    "Inductor may insert FP16 fallbacks for unsupported shapes. "
                    "Use torch._dynamo.explain() to inspect actual lowering."
                ),
            },
            "ops_quantized"   : "Conv2d + Linear (all ATen graph ops annotated by quantizer)",
            "calibration_samples": CALIB_SIZE,
            "accuracy"        : round(metrics["accuracy"],  6),
            "precision"       : round(metrics["precision"], 6),
            "recall"          : round(metrics["recall"],    6),
            "f1"              : round(metrics["f1"],        6),
            "accuracy_drop"   : round(baseline_acc - metrics["accuracy"], 6),
            "inference_latency": lat,
            "flops": {
                "total_flops"  : flops_fp32.get("total_flops"),
                "gflops"       : flops_fp32.get("gflops_fp32") or flops_fp32.get("gflops"),
                "note": (
                    "FLOPs identical to FP32 baseline. Inductor fuses quant+conv+dequant "
                    "into a single kernel where possible, reducing memory round-trips "
                    "but not the arithmetic op count. Use measured latency."
                ),
            },
        })
        print(f"           → Acc: {metrics['accuracy']:.4f}  "
              f"GPU: {lat['mean_ms']:.1f} ms")

    except ImportError as e:
        # PT2E not available on this PyTorch build — record skip, don't crash
        msg = str(e)
        print(f"           → SKIPPED (PT2E unavailable): {msg}")
        rows.append({
            "backend"    : "torch_pt2e_inductor_int8",
            "description": f"SKIPPED — PT2E not available on this PyTorch build: {msg}",
            "accuracy"   : None,
            "install_hint": "pip install torch>=2.3  (PT2E requires 2.3+)",
        })
    except Exception as e:
        print(f"           → FAILED: {e}")
        rows.append({"backend": "torch_pt2e_inductor_int8",
                     "description": f"FAILED: {e}", "accuracy": None})

    return rows


# ══════════════════════════════════════════════════════════════════════════════
# FP32 GPU Baseline
# ══════════════════════════════════════════════════════════════════════════════
def benchmark_fp32_gpu(fp32_model, test_loader, device, flops_fp32: Dict) -> Dict:
    """Measure FP32 GPU baseline for fair comparison."""
    print(f"\n  [0/3] FP32 GPU Baseline ({device})")
    model = copy.deepcopy(fp32_model).to(device).eval()
    metrics = evaluate_torch(model, test_loader, device)
    lat     = measure_gpu_throughput_ms(model, device)
    disk_mb = disk_size_mb(model.cpu())
    ram_mb  = sum(p.numel() for p in model.parameters()) * 4 / 1024 ** 2
    print(f"        Acc: {metrics['accuracy']:.4f}  "
          f"GPU: {lat['mean_ms']:.1f} ms  Disk: {disk_mb:.2f} MB")
    return {
        "accuracy"          : round(metrics["accuracy"],  6),
        "precision"         : round(metrics["precision"], 6),
        "recall"            : round(metrics["recall"],    6),
        "f1"                : round(metrics["f1"],        6),
        "size_disk_mb"      : round(disk_mb, 4),
        "ram_fp32_mb"       : round(ram_mb, 4),
        "inference_latency" : lat,
        "flops"             : flops_fp32,
        "device"            : str(device),
        "gpu_name"          : torch.cuda.get_device_name(device) if torch.cuda.is_available() else "N/A",
    }


# ══════════════════════════════════════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════════════════════════════════════
def main():
    parser = argparse.ArgumentParser(description="GPU PTQ for ResNet-50/CIFAR-10")
    parser.add_argument("--install-deps", action="store_true",
                        help="Install required GPU packages and exit")
    parser.add_argument("--device", default="cuda",
                        help="Target device: cuda, cuda:0, cuda:1, cpu")
    args, _ = parser.parse_known_args()  # ignore Jupyter kernel args (--f=...)

    if args.install_deps:
        install_deps()
        return

    # ── Dependency check ───────────────────────────────────────────────────────
    missing = []
    if torch is None:
        missing.append("torch  →  pip install torch --index-url https://download.pytorch.org/whl/cu121")
    if onnx is None:
        missing.append("onnx   →  pip install onnx")
    if ort is None:
        missing.append("onnxruntime-gpu  →  pip install onnxruntime-gpu")
    if missing:
        print("\n⚠  Missing required packages:")
        for m in missing:
            print(f"   {m}")
        print("\nRun:  python gpu_ptq.py --install-deps")
        sys.exit(1)

    device = torch.device(args.device if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*65}")
    print("  GPU Post-Training Quantization (PTQ) — ResNet-50 / CIFAR-10")
    print(f"  Device    : {device}")
    if torch.cuda.is_available():
        print(f"  GPU       : {torch.cuda.get_device_name(device)}")
        print(f"  CUDA      : {torch.version.cuda}")
        print(f"  cuDNN     : {torch.backends.cudnn.version()}")
    print("  Frameworks: ONNX Runtime GPU  |  TensorRT  |  PyTorch PT2E")
    print(f"{'='*65}")

    # ── Load baseline ──────────────────────────────────────────────────────────
    if not os.path.exists(BASELINE_MODEL_PATH):
        print(f"\n✗ Baseline model not found: {BASELINE_MODEL_PATH}")
        print("  Run the baseline training script first.")
        sys.exit(1)

    with open(BASELINE_METRICS_PATH) as f:
        baseline_metrics = json.load(f)
    baseline_acc = baseline_metrics["accuracy"]

    fp32_model   = load_baseline(BASELINE_MODEL_PATH)
    test_loader  = get_test_loader()
    calib_loader = get_calib_loader()

    # ── FLOPs (always computed on CPU model for consistency) ───────────────────
    # NOTE: FLOPs are IDENTICAL for INT8 and FP32 — quantization does not
    # change the arithmetic operation count. Only hardware throughput changes.
    print("\n  Computing FLOPs (op count — identical for FP32 and INT8)...")
    flops_fp32 = {}

    # Try torch.profiler first (native, no extra deps)
    try:
        flops_fp32 = count_flops_pytorch_profiler(fp32_model.cpu(), device="cpu")
        gf = flops_fp32.get("gflops_fp32", 0)
        pm = flops_fp32.get("params_M", 0)
        print(f"  FLOPs (torch.profiler): {gf:.3f} GFLOPs | {pm:.1f}M params")
        print("  Note: FLOPs are unchanged by quantization. "
              "INT8 speedup = hardware throughput (SIMD width), not fewer ops.")
    except Exception as e:
        print(f"  torch.profiler failed: {e}")
        if fvcore is not None:
            try:
                flops_fp32 = count_flops_fvcore(fp32_model.cpu())
                print(f"  FLOPs (fvcore fallback): {flops_fp32['gflops_fp32']:.3f} GFLOPs")
            except Exception as e2:
                print(f"  fvcore also failed: {e2}")
                flops_fp32 = count_flops_manual(fp32_model.cpu())
        else:
            flops_fp32 = count_flops_manual(fp32_model.cpu())
        gf = flops_fp32.get("gflops_fp32") or flops_fp32.get("gflops", 0)
        print(f"  FLOPs (manual fallback): {gf:.3f} GFLOPs")

    base_disk = disk_size_mb(fp32_model)

    # ── FP32 GPU Baseline ──────────────────────────────────────────────────────
    fp32_gpu_baseline = benchmark_fp32_gpu(fp32_model, test_loader, device, flops_fp32)

    # ── Run PTQ backends ───────────────────────────────────────────────────────
    all_results = []
    all_results.extend(
        run_onnx_ptq(fp32_model, test_loader, calib_loader,
                     baseline_acc, base_disk, device, flops_fp32)
    )
    all_results.append(
        run_tensorrt_ptq(fp32_model, test_loader, calib_loader,
                         baseline_acc, base_disk, device, flops_fp32)
    )
    all_results.extend(
        run_torch_cuda_ptq(fp32_model, test_loader, calib_loader,
                           baseline_acc, base_disk, device, flops_fp32)
    )

    # ── Best config ────────────────────────────────────────────────────────────
    valid = [r for r in all_results if r.get("accuracy") is not None]
    best  = min(valid, key=lambda r: (r.get("accuracy_drop", 99),
                                       r.get("size_disk_mb", 99))) if valid else {}

    # ── Build report ───────────────────────────────────────────────────────────
    report = {
        "experiment"  : "GPU Post-Training Quantization (PTQ)",
        "description" : (
            "GPU-native PTQ using three frameworks: "
            "(1) ONNX Runtime GPU with QDQ format and 3 calibration methods, "
            "(2) TensorRT INT8 engine with entropy histogram calibration and layer fusion, "
            "(3) PyTorch CUDA: dynamic PTQ (bandwidth savings only) + PT2E Inductor INT8. "
            "FLOPs measured via torch.profiler, fvcore, or manual hook-based counting."
        ),
        "ptq_math": {
            "quantize"   : "Q(x) = clip(round(x / S) + Z, q_min, q_max)",
            "dequantize" : "x̂ = S · (Q(x) − Z)",
            "error"      : "ε = x − x̂,  |ε| ≤ S/2",
            "scale"      : "S = (x_max − x_min) / (2^b − 1)  [b=8 → 255 levels for UINT8]",
            "zero_point" : "Z = clip(round(−x_min / S), q_min, q_max)",
            "int8_levels": "2^8 − 1 = 255  (UINT8 activations: [0, 255])",
            "gpu_advantage_clarification": {
                "correct_framing": (
                    "INT8 does NOT reduce FLOPs. FLOPs = op count = graph property, "
                    "independent of datatype. The GPU speedup from INT8 is a hardware "
                    "property: INT8 Tensor Core lanes process 4× more elements per cycle "
                    "than FP32 lanes (Ampere). This is a throughput advantage, not a "
                    "reduction in arithmetic work."
                ),
                "int8_tensor_core_throughput_ampere": "~4× vs FP32 (A100, RTX 30xx)",
                "int8_tensor_core_throughput_hopper": "~8× vs FP32 (H100)",
                "memory_bandwidth_reduction"        : "~4× (1 byte INT8 vs 4 bytes FP32 per weight)",
                "layer_fusion_benefit"              : (
                    "TensorRT Conv+BN+ReLU fusion reduces memory round-trips by ~3×, "
                    "independent of INT8 — this is a separate source of speedup."
                ),
            },
        },
        "flops_methodology": {
            "tool"          : "torch.profiler (primary), fvcore, or manual hooks (fallback)",
            "formula_conv"  : "FLOPs = 2 × (Cin/groups) × Cout × Kh × Kw × Hout × Wout",
            "formula_linear": "FLOPs = 2 × in_features × out_features",
            "formula_bn"    : "excluded — negligible vs Conv, fused by TRT/ORT, omitted by fvcore/thop",
            "macs_to_flops" : "FLOPs = 2 × MACs  (one multiply + one accumulate per MAC)",
            "critical_note" : (
                "FLOPs are IDENTICAL for INT8 and FP32 models. "
                "Quantization changes the storage dtype and enables wider SIMD execution; "
                "it does not change the number of arithmetic operations in the graph. "
                "Reporting 'INT8 FLOPs = FP32 FLOPs / 4' is incorrect — "
                "use measured wall-clock latency or throughput (imgs/sec) instead."
            ),
        },
        "gpu_info": {
            "device"    : str(device),
            "gpu_name"  : fp32_gpu_baseline.get("gpu_name", "N/A"),
            "cuda_ver"  : getattr(torch.version, "cuda", "N/A") if torch else "N/A",
            "cudnn_ver" : str(torch.backends.cudnn.version()) if torch and torch.cuda.is_available() else "N/A",
        },
        "baseline_fp32"     : baseline_metrics,
        "baseline_fp32_gpu" : fp32_gpu_baseline,
        "calibration_config": {
            "calib_size"   : CALIB_SIZE,
            "calib_batches": CALIB_BATCHES,
            "dataset"      : "CIFAR-10 training split (first 1024 images)",
        },
        "best_config" : {
            "backend"          : best.get("backend"),
            "calibration_method": best.get("calibration_method", best.get("observer")),
            "accuracy"         : best.get("accuracy"),
            "accuracy_drop"    : best.get("accuracy_drop"),
            "size_disk_mb"     : best.get("size_disk_mb"),
            "compression_ratio": best.get("compression_ratio"),
            "latency_mean_ms"  : best.get("inference_latency", {}).get("mean_ms"),
        } if best else {},
        "results"     : all_results,
        "framework_notes": {
            "onnxruntime_gpu": (
                "Requires onnxruntime-gpu package (not onnxruntime). "
                "CUDAExecutionProvider dispatches quantized ops to: "
                "cuDNN INT8 (QLinearConv), cuBLASLt INT8 (QLinearMatMul), "
                "custom CUDA kernels (QLinearAdd, etc.). "
                "Kernel selection is determined by ORT's graph partitioner at session "
                "creation — not every op is guaranteed to use cuDNN INT8. "
                "QDQ format inserts QuantizeLinear/DequantizeLinear pairs; "
                "no engine compilation step (faster to deploy than TRT)."
            ),
            "tensorrt": (
                "Highest throughput for production GPU serving. "
                "Kernel autotuning at build time (GPU-specific binary — NOT portable). "
                "IInt8EntropyCalibrator2 uses histogram-based KL threshold search "
                "(heuristic, not analytic argmin). "
                "Layer fusion (Conv+BN+ReLU) is a separate speedup from INT8. "
                "Requires pycuda for Python bindings."
            ),
            "torch_pt2e": (
                "PT2E (PyTorch 2 Export): torch.export / capture_pre_autograd_graph → "
                "quantizer annotations → prepare_pt2e → calibrate → convert_pt2e → "
                "torch.compile(backend='inductor'). "
                "X86InductorQuantizer is named for CPU history but also annotates GPU ops; "
                "Inductor selects INT8 Triton/cuBLAS kernels at compile time. "
                "CUDA INT8 via PT2E is still maturing (torch 2.3–2.5); "
                "some ops may fall back to FP16. Check torch._dynamo.explain()."
            ),
            "torch_dynamic_gpu": (
                "PyTorch dynamic PTQ on CUDA stores Linear weights as INT8 but "
                "dequantizes them to FP16/FP32 before the cuBLAS GEMM kernel. "
                "This saves memory bandwidth (~2× for Linear), NOT compute. "
                "It is NOT equivalent to true INT8 GEMM (TensorRT / PT2E)."
            ),
            "why_not_cpu_static": (
                "torch.quantization static PTQ (prepare/convert + eager mode) is CPU-only. "
                "For GPU INT8: use ORT-GPU, TensorRT, or PT2E+Inductor."
            ),
            "flops_vs_throughput": (
                "CRITICAL DISTINCTION: "
                "FLOPs (op count) are IDENTICAL for INT8 and FP32 models. "
                "The INT8 GPU speedup is entirely a hardware throughput property: "
                "Tensor Core SIMD width, memory bandwidth, and kernel fusion. "
                "Never divide FP32 FLOPs by 4 to get 'INT8 FLOPs' — "
                "that conflates operation count with hardware execution rate."
            ),
        },
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)

    print(f"\n{'='*65}")
    print(f"  ✓ Saved → {OUTPUT_JSON}")
    if best:
        print(f"  Best    : {best.get('backend')}  "
              f"acc={best.get('accuracy', 'N/A'):.4f}  "
              f"drop={best.get('accuracy_drop', 'N/A'):+.4f}")
        lat = best.get("inference_latency", {})
        if lat.get("mean_ms"):
            fp32_lat = fp32_gpu_baseline.get("inference_latency", {}).get("mean_ms", 1)
            speedup  = fp32_lat / lat["mean_ms"] if lat["mean_ms"] else 0
            print(f"  Speedup : {speedup:.2f}× vs FP32 GPU  "
                  f"({lat['mean_ms']:.1f} ms vs {fp32_lat:.1f} ms)")
    print(f"{'='*65}\n")


if __name__ == "__main__":
    main()


  GPU Post-Training Quantization (PTQ) — ResNet-50 / CIFAR-10
  Device    : cuda
  GPU       : NVIDIA GeForce RTX 5070 Laptop GPU
  CUDA      : 12.8
  cuDNN     : 92000
  Frameworks: ONNX Runtime GPU  |  TensorRT  |  PyTorch PT2E

  Computing FLOPs (op count — identical for FP32 and INT8)...
  torch.profiler failed: name 'count_flops_pytorch_profiler' is not defined
  FLOPs (manual fallback): 2.596 GFLOPs

  [0/3] FP32 GPU Baseline (cuda)


W0428 23:37:40.019000 35396 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


        Acc: 0.9320  GPU: 51.2 ms  Disk: 90.05 MB

  [1/3] ONNX Runtime — Static INT8 PTQ
        Framework: onnxruntime-gpu (CUDAExecutionProvider)
        Kernels:   cuDNN INT8 (Conv), cuBLASLt (MatMul), custom (Add)
        S/Z:       calibrated once from calib data; frozen in graph
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "e:\baseline_resnet50_cifar10\env\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "e:\baseline_resnet50_cifar10\env\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "e:\baseline_resnet50_cifar10\env\Lib\site-packages\onnxscript\version_converter\__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "e:\baseline_resnet50_cifar10\env\Lib\site-packages\onnx\version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: D:\a\onnx\onnx\onnx/version

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
        ✓ ONNX exported → resnet50_fp32.onnx  (0.22 MB, opset=17)
        Calibration: MinMax      

 → Acc: 0.9319  Drop: +0.0001  Disk: 90.18 MB  Lat: 65.0 ms (CUDAExecutionProvider)
        Calibration: Entropy     

 → FAILED: [ONNXRuntimeError] : 6 : RUNTIME_EXCEPTION : Non-zero status code returned while running Conv node. Name:'node_Conv_772' Status Message: bad allocation
        Calibration: Percentile  

 → FAILED: [ONNXRuntimeError] : 6 : RUNTIME_EXCEPTION : Non-zero status code returned while running Conv node. Name:'node_Conv_782' Status Message: bad allocation

  [2/3] TensorRT — INT8 PTQ
        ⚠ TensorRT/pycuda not available: No module named 'tensorrt'
          Install: pip install tensorrt pycuda

  [3/3] PyTorch CUDA — Dynamic PTQ + PT2E Inductor INT8
        3a. Dynamic PTQ (Linear → qint8) on CUDA
           → FAILED: Could not run 'quantized::linear_dynamic' with arguments from the 'CUDA' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'quantized::linear_dynamic' is only available for these backends: [CPU, Meta, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradO